# Phase 6B — PostgreSQL Persistence

## Overview

Phase 6B introduced persistent database storage for the document-intelligence system.

The main objectives were to:

* install and configure PostgreSQL,
* connect the application using SQLAlchemy and Psycopg,
* define the database schema,
* persist processed documents and analysis results,
* store human-review actions,
* store audit events,
* verify transaction integrity and database relationships.

---

# 1. PostgreSQL Environment Setup

PostgreSQL 18.6 was installed and configured locally.

The database server was verified using the PostgreSQL command-line client.

A dedicated project database was created:

```text
vigilox_document_intelligence
```

The application connected to PostgreSQL through a `DATABASE_URL` stored in the environment configuration.

Database credentials remained outside source control.

---

# 2. Database Connectivity

The Python application connected to PostgreSQL using:

* SQLAlchemy 2.x
* Psycopg 3

A reusable SQLAlchemy engine and session factory were introduced.

The database connection was validated successfully with a direct query.

This established the persistence foundation required by the remaining backend phases.

---

# 3. Database Schema

Four main tables were introduced.

```text
documents
document_analyses
human_reviews
audit_events
```

The relationship structure became:

```text
documents
    │
    ├── document_analyses
    ├── human_reviews
    └── audit_events
```

---

# 4. Documents Table

The `documents` table stores the primary identity and metadata of every processed document.

Important information includes:

* unique document ID,
* original filename,
* content type,
* detected document type,
* processing status,
* creation timestamp,
* update timestamp.

This table acts as the parent record for the complete document lifecycle.

---

# 5. Document Analysis Storage

The `document_analyses` table stores the complete machine-generated analysis.

Stored information includes:

* structured extraction,
* OCR lines,
* OCR evidence,
* evidence-validation flags,
* field confidence,
* date and logical validation,
* anomaly validation,
* machine review decision.

The large and evolving pipeline output was stored using PostgreSQL `JSONB`.

This provided flexibility while preserving structured query capability.

---

# 6. Relational and JSONB Design

A hybrid persistence strategy was used.

```text
Stable searchable metadata
        ↓
Relational columns

Large nested analysis data
        ↓
JSONB
```

This avoided creating excessive relational tables for every OCR line or validation component while still allowing PostgreSQL queries on important nested values.

---

# 7. Repository Layer

A repository layer was introduced between the application logic and ORM models.

Repositories were created for:

* documents,
* document analyses,
* human reviews,
* audit events.

This keeps database operations separate from OCR, extraction, validation, and API logic.

The design improves maintainability and reduces direct database coupling across the application.

---

# 8. Persistence Service

A persistence service was introduced to coordinate database transactions.

For processed documents, the persistence workflow became:

```text
Document Metadata
        +
Complete Analysis
        +
Machine Review Audit
        ↓
Single Database Transaction
```

If any part of the transaction fails, the complete operation is rolled back.

This prevents partially stored document records.

---

# 9. Document and Analysis Persistence Test

A deterministic pipeline-shaped result was first stored in PostgreSQL.

The test verified that:

* a document row was created,
* an analysis row was created,
* UUID identifiers were generated,
* nested JSON data was stored successfully,
* the persisted result could be retrieved from a new database session,
* stored values matched the original input.

This confirmed that persistence was occurring in PostgreSQL rather than only remaining in application memory.

---

# 10. Real Pipeline Persistence

The actual document-intelligence pipeline was then connected to the persistence layer.

A real guard licence passed through:

```text
Image
  ↓
OCR
  ↓
LLM Extraction
  ↓
Evidence Validation
  ↓
Confidence
  ↓
Date Validation
  ↓
Anomaly Detection
  ↓
Review Decision
  ↓
PostgreSQL Persistence
```

The real pipeline result was stored successfully.

The persisted record contained:

* 24 OCR lines,
* correct document classification,
* licence information,
* anomaly results,
* machine review decision.

The stored result was retrieved successfully from a new database session.

---

# 11. JSONB Query Verification

Nested analysis values were queried directly from PostgreSQL.

The database correctly returned:

* document type,
* review decision,
* review priority.

This demonstrated that storing the analysis as JSONB still allowed direct structured querying.

---

# 12. Human Review Persistence

Human-review results were integrated into PostgreSQL.

Supported human actions remained:

```text
APPROVE
REJECT
CORRECT
```

Stored review information includes:

* reviewer ID,
* machine decision,
* machine priority,
* machine reason codes,
* human action,
* corrections,
* notes,
* review timestamp.

---

# 13. Correction Provenance

Human corrections were intentionally stored separately from the original machine extraction.

Example:

```text
Machine expiry date:
2026-01-01

Human correction:
2027-01-01
```

The original machine analysis remained unchanged.

The human correction was stored separately in the `human_reviews` table.

This preserves:

* machine evidence,
* human intervention,
* correction provenance,
* research traceability.

---

# 14. Audit Persistence

Persistent audit events were added for both machine and human actions.

Two major event types were stored:

```text
MACHINE_REVIEW_DECISION
HUMAN_REVIEW
```

Machine audit events record:

* decision,
* review-required state,
* priority,
* reason codes.

Human audit events record:

* reviewer,
* action,
* corrections,
* notes,
* machine decision context,
* review timestamp.

---

# 15. Transaction Safety

Critical related operations were stored within single database transactions.

For machine processing:

```text
Document
+
Analysis
+
Machine Audit
      ↓
One Transaction
```

For human review:

```text
Human Review
+
Human Audit
      ↓
One Transaction
```

This ensures that related records are either fully committed or fully rolled back.

---

# 16. Orphan Record Prevention

The persistence layer verifies that a document exists before accepting a human review.

A review for a missing document was intentionally tested.

The system correctly rejected the operation.

The test confirmed that:

* no orphan review was created,
* no orphan audit event was created.

---

# 17. Foreign-Key and Cascade Behavior

Foreign-key relationships were configured between the parent document and dependent records.

Deleting a document successfully removed associated:

* analysis,
* human review,
* audit events.

This was verified through a cascade-delete test.

---

# 18. Final Database Integrity Test

The final Phase 6B verification created a complete database state containing:

```text
1 Document
1 Analysis
1 Human Review
2 Audit Events
```

The final test verified:

* correct row counts,
* JSONB querying,
* document relationships,
* review persistence,
* audit persistence,
* orphan prevention,
* cascade deletion.

All integrity checks passed.

---

# Key Results

Phase 6B successfully demonstrated that the system can persist the complete document lifecycle.

The database now supports:

* document metadata,
* full machine analysis,
* real OCR and LLM results,
* machine review decisions,
* human review actions,
* corrections,
* persistent audit history,
* relational integrity,
* transaction safety.

A key design result is that human corrections do not overwrite machine-generated evidence.

Instead, both machine output and human intervention remain independently traceable.

---

# Final Architecture

```text
Document Pipeline
       ↓
Persistence Service
       ↓
PostgreSQL
       │
       ├── documents
       │
       ├── document_analyses
       │
       ├── human_reviews
       │
       └── audit_events
```

Human-review flow:

```text
Machine Review Decision
        ↓
Human Reviewer
        ↓
APPROVE / REJECT / CORRECT
        ↓
human_reviews
        +
audit_events
```

---

# Final Conclusion

Phase 6B successfully transformed the document-intelligence system from a temporary request-based workflow into a persistent backend system.

The project now has a verified PostgreSQL persistence layer with repository abstraction, transaction management, JSONB analysis storage, human-review persistence, audit history, and database integrity controls.

**Phase 6B — PostgreSQL Persistence: Complete**
